# **Notebook: Add county information to `solar_lcoe_ReEDS.csv`**

This notebook adds county information to the solar metadata file `solar_lcoe_ReEDS.csv` using a polygon-to-polygon matching workflow, then filters the final outputs to the requested WECC states.

The process is:
1. load `solar_lcoe_ReEDS.csv`
2. load the solar CPA polygons from the Zenodo candidate project area shapefiles
3. load the TIGER/Line county shapefile
4. spatially match each CPA polygon to the counties it intersects
5. for CPAs that cross multiple counties, assign a primary county using the largest overlap area
6. merge the county assignment back into `solar_lcoe_ReEDS.csv`
7. filter the final outputs to the requested WECC states
8. save the updated metadata file and QA outputs

The final result is a WECC-filtered, county-enriched solar metadata file that can be used in the next clustering step.

## **Folder structure expected by this notebook**

Place the input files here inside the repo:

```text
solar-county-analysis/
  data/
    reeds_county_mapping/
      inputs/
        solar_lcoe_ReEDS.csv
        CandidateProjectAreas_WindAndSolar_20210623/
          CandidateProjectArea_SolarPV.shp
          CandidateProjectArea_SolarPV.shx
          CandidateProjectArea_SolarPV.dbf
          CandidateProjectArea_SolarPV.prj
          ...
        tl_2024_us_county/
          tl_2024_us_county.shp
          tl_2024_us_county.shx
          tl_2024_us_county.dbf
          tl_2024_us_county.prj
          ...
      outputs/
  notebooks/
    reeds_add_counties_to_solar_lcoe.ipynb

In [22]:
from pathlib import Path
import json
import pandas as pd
import geopandas as gpd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)
pd.set_option("display.max_colwidth", 160)

In [23]:
NOTEBOOK_DIR = Path(".").resolve()
REPO_ROOT = NOTEBOOK_DIR.parent

DATA_DIR = REPO_ROOT / "data" / "reeds_county_mapping"
INPUT_DIR = DATA_DIR / "inputs"
OUTPUT_DIR = DATA_DIR / "outputs"

SOLAR_METADATA_CSV = INPUT_DIR / "solar_lcoe_ReEDS.csv"

CPA_DIR = INPUT_DIR / "CandidateProjectAreas_WindAndSolar_20210623"
CPA_SHP = CPA_DIR / "CandidateProjectArea_SolarPV.shp"

COUNTY_DIR = INPUT_DIR / "tl_2024_us_county"
COUNTY_SHP = COUNTY_DIR / "tl_2024_us_county.shp"

COUNTY_GEOJSON_OPTIONAL = REPO_ROOT / "county_layer_for_gui.geojson"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

paths_to_check = {
    "repo_root": REPO_ROOT,
    "data_dir": DATA_DIR,
    "solar_metadata_csv": SOLAR_METADATA_CSV,
    "cpa_shapefile": CPA_SHP,
    "county_shapefile": COUNTY_SHP,
    "county_geojson_optional": COUNTY_GEOJSON_OPTIONAL,
    "output_dir": OUTPUT_DIR,
}

for name, p in paths_to_check.items():
    print(f"{name}: {p.exists()} -> {p}")

repo_root: True -> /Users/laurenvo/Documents/github/solar-county-propensity-scores
data_dir: True -> /Users/laurenvo/Documents/github/solar-county-propensity-scores/data/reeds_county_mapping
solar_metadata_csv: True -> /Users/laurenvo/Documents/github/solar-county-propensity-scores/data/reeds_county_mapping/inputs/solar_lcoe_ReEDS.csv
cpa_shapefile: True -> /Users/laurenvo/Documents/github/solar-county-propensity-scores/data/reeds_county_mapping/inputs/CandidateProjectAreas_WindAndSolar_20210623/CandidateProjectArea_SolarPV.shp
county_shapefile: True -> /Users/laurenvo/Documents/github/solar-county-propensity-scores/data/reeds_county_mapping/inputs/tl_2024_us_county/tl_2024_us_county.shp
county_geojson_optional: True -> /Users/laurenvo/Documents/github/solar-county-propensity-scores/county_layer_for_gui.geojson
output_dir: True -> /Users/laurenvo/Documents/github/solar-county-propensity-scores/data/reeds_county_mapping/outputs


## Load the three main input datasets

This cell loads the three core datasets used in the notebook.

**1. `solar_meta`**
- Loaded from `solar_lcoe_ReEDS.csv`
- This is the main solar metadata table being updated in the notebook
- It contains one row per solar Candidate Project Area, or CPA
- The goal of the notebook is to add county information into this table so it can later be used for county-based clustering

**2. `cpa`**
- Loaded from `CandidateProjectArea_SolarPV.shp`
- This is the solar CPA shapefile from the Zenodo candidate project area dataset
- Each row is a geographic polygon showing the actual shape and location of a solar Candidate Project Area
- This dataset is what makes it possible to connect each `CPA_ID` to a real location on the map

**3. `counties`**
- Loaded from `tl_2024_us_county.shp`
- This is the county boundary shapefile from TIGER/Line
- Each row is a county polygon
- This dataset is used to determine which county each CPA belongs to

In [ ]:
solar_meta = pd.read_csv(SOLAR_METADATA_CSV, low_memory=False)
cpa = gpd.read_file(CPA_SHP)
counties = gpd.read_file(COUNTY_SHP)

# QA / inspection prints
print("solar_meta shape:", solar_meta.shape)
print("solar_meta columns:", solar_meta.columns.tolist())
print("\ncpa shape:", cpa.shape)
print("cpa columns:", cpa.columns.tolist())
print("cpa CRS:", cpa.crs)
print("\ncounties shape:", counties.shape)
print("counties columns:", counties.columns.tolist())
print("counties CRS:", counties.crs)

display(solar_meta.head())
display(cpa.head())
display(counties.head())

solar_meta shape: (405737, 43)
solar_meta columns: ['Area', 'd_trans', 'd_sub', 'd_road', 'd_load_750', 'd_existing', 'd_plannedF', 'm_slope', 'm_popden', 'm_HMI', 'm_primeFarmland', 'incap', 'CPA_ID', 'm_aspect', 'm_aspect_min', 'Shape_Leng', 'm_landcover', 'exFacil', 'plFacil', 'Qual_Coal', 'Qual_Emp', 'Qual_Brown', 'anyQual', 'Qual_noBF', 'SocialImpa', 'EnviroImpa', 'Shape_Le_1', 'Shape_Area', 'pop_density_bin', 'tech', 'metro_id', 'metro_region', 'cpa_mw', 'cf', 'path', 'resource_annuity', 'resource_fom', 'interconnect_annuity', 'lcoe', 'interconnect_capex_mw', 'total_interconnect_km', 'offshore_interconnect_km', 'ipm_region']

cpa shape: (406110, 22)
cpa columns: ['Area', 'd_trans', 'd_sub', 'd_road', 'd_load_750', 'd_existing', 'd_plannedF', 'm_slope', 'm_popden', 'm_HMI', 'm_primeFar', 'incap', 'CPA_ID', 'm_aspect', 'm_aspect_m', 'Shape_Leng', 'Shape_Area', 'm_landcove', 'exFacil', 'plFacil', 'cf', 'geometry']
cpa CRS: PROJCS["NAD_1983_Albers",GEOGCS["NAD83",DATUM["North_America

,Area,d_trans,d_sub,d_road,d_load_750,d_existing,d_plannedF,m_slope,m_popden,m_HMI,m_primeFarmland,incap,CPA_ID,m_aspect,m_aspect_min,Shape_Leng,m_landcover,exFacil,plFacil,Qual_Coal,Qual_Emp,Qual_Brown,anyQual,Qual_noBF,SocialImpa,EnviroImpa,Shape_Le_1,Shape_Area,pop_density_bin,tech,metro_id,metro_region,cpa_mw,cf,path,resource_annuity,resource_fom,interconnect_annuity,lcoe,interconnect_capex_mw,total_interconnect_km,offshore_interconnect_km,ipm_region
0,2.00,19.142255,26.470939,0.000000,158.648693,265.038776,246.071077,0.125000,17.375000,0.780994,0.0,18.000001,1,48.625000,-1.0,6000.000177,81,0,0,0,1,0,1,1,3.906250,1.906250,6000.0,2000000.0,pop_den_(10-30],photovoltaic,42660,p1,4.500000,0.220500,"[1, 200355]",48359.91,15221.571,23691.982,45.182354,497798.70,143.821594,0.0,p1
1,3.75,13.030586,20.955390,1.085683,156.003598,264.714080,245.267324,1.600000,34.266667,0.786600,1.0,33.749998,2,145.866667,-1.0,8999.999866,81,0,0,0,1,0,1,1,-2.000000,5.250000,9000.0,3750000.0,pop_den_(30-40],photovoltaic,42660,p1,4.218750,0.224434,"[2, 200355]",48359.91,15221.571,23711.625,44.400440,498211.40,140.985809,0.0,p1
2,3.00,35.946548,45.514552,6.423029,156.322476,252.099160,235.024352,9.833333,0.000000,0.097131,0.0,26.999998,3,237.166667,129.0,10999.999725,42,0,0,0,1,0,1,1,-5.000000,1.000000,11000.0,3000000.0,pop_den_(0-5],photovoltaic,42660,p1,26.999998,0.222281,"[3, 200355]",48359.91,15221.571,25068.760,45.527447,526726.56,150.642654,0.0,p1
3,2.00,22.097498,30.668918,0.000000,149.529784,252.050029,233.665701,3.375000,35.125000,0.544276,1.0,18.000000,4,215.750000,-1.0,9000.000066,81,0,0,0,1,0,1,1,10.468750,9.593750,9000.0,2000000.0,pop_den_(30-40],photovoltaic,42660,p1,2.250000,0.223767,"[4, 200355]",48359.91,15221.571,22436.877,43.882450,471427.34,135.036041,0.0,p1
4,2.25,18.546952,28.017060,0.126662,145.881474,249.425370,230.785601,8.777778,1.888889,0.429502,0.0,20.250001,6,201.777778,7.0,8000.000037,42,0,0,0,1,0,1,1,9.583333,18.055556,8000.0,2250000.0,pop_den_(0-5],photovoltaic,42660,p1,20.250000,0.223767,"[6, 200355]",48359.91,15221.571,21848.154,43.582115,459057.50,130.457626,0.0,p1


,Area,d_trans,d_sub,d_road,d_load_750,d_existing,d_plannedF,m_slope,m_popden,m_HMI,m_primeFar,incap,CPA_ID,m_aspect,m_aspect_m,Shape_Leng,Shape_Area,m_landcove,exFacil,plFacil,cf,geometry
0,2.00,19.142255,26.470939,0.000000,158.648693,265.038776,246.071077,0.125000,17.375000,0.780994,0.0,18.000001,1,48.625000,-1.0,6000.000177,2.000000e+06,81,0,0,0.220263,"POLYGON ((-1924591.227 3153922.078, -1924591.227 3153422.078, -1924091.227 3153422.078, -1924091.227 3152422.078, -1925591.227 3152422.078, -1925591.227 315..."
1,3.75,13.030586,20.955390,1.085683,156.003598,264.714080,245.267324,1.600000,34.266667,0.786600,1.0,33.749998,2,145.866667,-1.0,8999.999866,3.750000e+06,81,0,0,0.224177,"POLYGON ((-1929591.227 3151922.078, -1929591.227 3151422.078, -1929091.227 3151422.078, -1929091.227 3150422.078, -1929591.227 3150422.078, -1929591.227 314..."
2,3.00,35.946548,45.514552,6.423029,156.322476,252.099160,235.024352,9.833333,0.000000,0.097131,0.0,26.999998,3,237.166667,129.0,10999.999725,3.000000e+06,42,0,0,0.222078,"POLYGON ((-1900091.227 3148922.078, -1900091.227 3148422.078, -1899591.227 3148422.078, -1899591.227 3146922.078, -1900591.227 3146922.078, -1900591.227 314..."
3,2.00,22.097498,30.668918,0.000000,149.529784,252.050029,233.665701,3.375000,35.125000,0.544276,1.0,18.000000,4,215.750000,-1.0,9000.000066,2.000000e+06,81,0,0,0.223520,"POLYGON ((-1914091.227 3143922.078, -1914091.227 3142922.078, -1913591.227 3142922.078, -1913591.227 3142422.078, -1915091.227 3142422.078, -1915091.227 314..."
4,2.25,18.546952,28.017060,0.126662,145.881474,249.425370,230.785601,8.777778,1.888889,0.429502,0.0,20.250001,6,201.777778,7.0,8000.000037,2.250000e+06,42,0,0,0.223520,"POLYGON ((-1915591.227 3139422.078, -1915591.227 3138922.078, -1916591.227 3138922.078, -1916591.227 3139422.078, -1917091.227 3139422.078, -1917091.227 313..."


,STATEFP,COUNTYFP,COUNTYNS,GEOID,GEOIDFQ,NAME,NAMELSAD,LSAD,CLASSFP,MTFCC,CSAFP,CBSAFP,METDIVFP,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,geometry
0,31,039,00835841,31039,0500000US31039,Cuming,Cuming County,06,H1,G4020,NaN,NaN,NaN,A,1477563042,10772508,+41.9158651,-096.7885168,"POLYGON ((-96.55525 41.82892, -96.55524 41.82758, -96.55524 41.82753, -96.55524 41.82739, -96.55524 41.8243, -96.55523 41.82217, -96.55524 41.82037, -96.555..."
1,53,069,01513275,53069,0500000US53069,Wahkiakum,Wahkiakum County,06,H1,G4020,NaN,NaN,NaN,A,680980773,61564428,+46.2946377,-123.4244583,"POLYGON ((-123.72755 46.2645, -123.72756 46.26476, -123.72768 46.27377, -123.72773 46.27788, -123.72774 46.27872, -123.72783 46.28508, -123.72788 46.28834, ..."
2,35,011,00933054,35011,0500000US35011,De Baca,De Baca County,06,H1,G4020,NaN,NaN,NaN,A,6016818941,29090018,+34.3592729,-104.3686961,"POLYGON ((-104.89337 34.08894, -104.89337 34.08908, -104.89334 34.09434, -104.89334 34.09458, -104.89334 34.09481, -104.89331 34.10002, -104.89329 34.10286,..."
3,31,109,00835876,31109,0500000US31109,Lancaster,Lancaster County,06,H1,G4020,339,30700,NaN,A,2169269508,22850511,+40.7835474,-096.6886584,"POLYGON ((-96.68493 40.5233, -96.69219 40.52312, -96.69369 40.52309, -96.6944 40.52307, -96.69461 40.52307, -96.70282 40.52307, -96.7029 40.52307, -96.70418..."
4,31,129,00835886,31129,0500000US31129,Nuckolls,Nuckolls County,06,H1,G4020,NaN,NaN,NaN,A,1489645201,1718484,+40.1764918,-098.0468422,"POLYGON ((-98.2737 40.1184, -98.27374 40.1224, -98.27374 40.12253, -98.27375 40.12314, -98.27378 40.12501, -98.2737 40.13304, -98.27369 40.13943, -98.27369 ..."


## Notes on the matching logic

`solar_lcoe_ReEDS.csv` already includes a unique `CPA_ID` for each solar Candidate Project Area record. The Zenodo solar shapefile also contains `CPA_ID`, which is the key used to connect the metadata table to the CPA polygons.

The county shapefile contains county boundary polygons and identifier fields such as:
- `GEOID` = county FIPS code
- `NAME` = county name
- `NAMELSAD` = county name with legal/statistical area description
- `STATEFP` = state FIPS code

The goal of this workflow is to assign a **primary county** to each CPA so the enriched metadata file can be used for county-based clustering later in PowerGenome / Switch-PG-ReEDS. Because a single CPA can cross more than one county boundary, the notebook does not assume a one-to-one match immediately. Instead, it first finds **all counties each CPA intersects**, then calculates how much of the CPA overlaps each county, and finally assigns the CPA to the county with the **largest overlap area**.

## Reduce the geospatial layers to only the fields needed

The full CPA shapefile and full county shapefile contain many extra columns that are not needed for the county-assignment workflow. To make the process easier to follow and more efficient to run, the notebook keeps only the minimum fields needed.

For the CPA layer, it keeps:
- `CPA_ID`
- `geometry`

`CPA_ID` is the key that links the CPA polygon back to `solar_lcoe_ReEDS.csv`, and `geometry` is the actual map shape of the solar Candidate Project Area.

For the county layer, it keeps:
- `GEOID`
- `NAME`
- `NAMELSAD`
- `STATEFP`
- `geometry`

These fields are enough to identify the county after the spatial join and preserve the key geographic identifiers needed for QA and downstream use.

The county fields are then renamed into clearer names for the rest of the notebook:
- `county_fips`
- `county_name`
- `county_name_full`
- `state_fips`

This makes the later merge steps and final outputs easier to read.

## Reproject both layers before calculating overlap area

The CPA and county shapefiles may originally be stored in a geographic CRS such as `EPSG:4326`, where coordinates are represented as longitude and latitude. That is fine for displaying features on a map, but it is not the best format for measuring polygon area.

Because this workflow assigns each CPA to the county with the **largest overlap area**, the area calculations need to be done in a projected coordinate system rather than in latitude/longitude degrees.

The notebook reprojects both the CPA polygons and the county polygons to `EPSG:5070`, which is a U.S. Albers equal-area projection. This is a good choice because:
- it is designed for use across the United States
- area calculations are much more meaningful
- it keeps the CPA-to-county overlap comparisons consistent across the study area

This reprojection does **not** change what the polygons represent. It only changes the coordinate system used internally so the overlap calculations are accurate.

In [ ]:
cpa = cpa[["CPA_ID", "geometry"]].copy()

counties = counties[["GEOID", "NAME", "NAMELSAD", "STATEFP", "geometry"]].copy()

counties = counties.rename(columns={
    "GEOID": "county_fips",
    "NAME": "county_name",
    "NAMELSAD": "county_name_full",
    "STATEFP": "state_fips",
})

cpa = cpa.to_crs("EPSG:5070")
counties = counties.to_crs("EPSG:5070")

print("CPA CRS after reprojection:", cpa.crs)
print("County CRS after reprojection:", counties.crs)
print("Unique CPA_IDs in metadata:", solar_meta["CPA_ID"].nunique())
print("Unique CPA_IDs in shapefile:", cpa["CPA_ID"].nunique())

CPA CRS after reprojection: EPSG:5070
County CRS after reprojection: EPSG:5070
Unique CPA_IDs in metadata: 405737
Unique CPA_IDs in shapefile: 406110


## Spatially match each CPA polygon to the counties it intersects

This cell performs the first actual geographic matching step in the notebook.

At this point:
- `cpa` contains the solar Candidate Project Area polygons from the Zenodo shapefile
- `counties` contains the county boundary polygons from the TIGER/Line shapefile

The purpose of this step is **not** to choose the final county for each CPA yet. Instead, the goal is to find **all possible county matches** for each CPA based on geographic overlap.

The spatial join uses `gpd.sjoin(..., predicate="intersects")`. This means the notebook checks each CPA polygon against the county polygons and keeps every county that the CPA touches or overlaps.

This is important because a single CPA can cross county boundaries. If the notebook forced each CPA into just one county too early, it could make the wrong assignment before checking how much of the CPA actually falls inside each county.

The result is an intermediate table called `cpa_county_matches`.

## What `cpa_county_matches` represents

Each row in `cpa_county_matches` is one **CPA-to-county match**.

This means:
- one CPA may appear in multiple rows
- each row represents one county that intersects that CPA
- this table is the full set of candidate county matches before the notebook applies the largest-overlap rule

This intermediate table is the starting point for the later step where the notebook assigns one **primary county** to each CPA.

## What the displayed output shows

The printed shape shows the size of the intermediate match table.

For example, if the output shows something like:

- `cpa_county_matches shape: (470356, 7)`

that means:
- there are **470,356 CPA-to-county match rows**
- there are **7 columns** in this intermediate table

This row count is larger than the number of unique CPAs because some CPAs intersect more than one county.

## What the displayed columns mean

The first few rows of the table show the structure of this intermediate result:

- `CPA_ID`
  - the unique identifier for the Candidate Project Area

- `geometry`
  - the CPA polygon geometry
  - this is the actual mapped shape of the candidate solar area

- `index_right`
  - the row index of the matched county in the county GeoDataFrame
  - this is mainly an internal lookup field used to connect the CPA match back to the correct county polygon

- `county_fips`
  - the county FIPS code
  - this is the standard county identifier

- `county_name`
  - the short county name

- `county_name_full`
  - the fuller legal/statistical county name, such as “Whatcom County”

- `state_fips`
  - the state FIPS code for the matched county

## What the example rows mean

In the displayed rows, the first few CPAs are all matched to **Whatcom County** with county FIPS `53073` and state FIPS `53`.

This means those particular CPA polygons intersect Whatcom County. At this stage, that does **not** necessarily mean Whatcom County is the final county assignment for every case in the full dataset. It only means that these CPAs have been identified as intersecting that county.

The notebook still needs one more step after this:
- calculate how much of each CPA overlaps each matched county
- then choose the county with the **largest overlap area** as the primary county

In [ ]:
cpa_county_matches = gpd.sjoin(
    cpa,
    counties,
    how="left",
    predicate="intersects"
)

# Print the shape of the intermediate match table.
# If the number of rows is much larger than the number of CPAs, that is expected,
# because some CPAs will intersect more than one county.
print("cpa_county_matches shape:", cpa_county_matches.shape)

# At this stage, we expect to see:
# - CPA_ID from the CPA polygons
# - county fields from the county layer
# - an index_right column indicating which county polygon was matched
display(cpa_county_matches.head())

cpa_county_matches shape: (470356, 7)


,CPA_ID,geometry,index_right,county_fips,county_name,county_name_full,state_fips
0,1,"POLYGON ((-1924591.227 3153922.078, -1924591.227 3153422.078, -1924091.227 3153422.078, -1924091.227 3152422.078, -1925591.227 3152422.078, -1925591.227 315...",1739.0,53073,Whatcom,Whatcom County,53
1,2,"POLYGON ((-1929591.227 3151922.078, -1929591.227 3151422.078, -1929091.227 3151422.078, -1929091.227 3150422.078, -1929591.227 3150422.078, -1929591.227 314...",1739.0,53073,Whatcom,Whatcom County,53
2,3,"POLYGON ((-1900091.227 3148922.078, -1900091.227 3148422.078, -1899591.227 3148422.078, -1899591.227 3146922.078, -1900591.227 3146922.078, -1900591.227 314...",1739.0,53073,Whatcom,Whatcom County,53
3,4,"POLYGON ((-1914091.227 3143922.078, -1914091.227 3142922.078, -1913591.227 3142922.078, -1913591.227 3142422.078, -1915091.227 3142422.078, -1915091.227 314...",1739.0,53073,Whatcom,Whatcom County,53
4,6,"POLYGON ((-1915591.227 3139422.078, -1915591.227 3138922.078, -1916591.227 3138922.078, -1916591.227 3139422.078, -1917091.227 3139422.078, -1917091.227 313...",1739.0,53073,Whatcom,Whatcom County,53


## Calculate CPA-to-county overlap metrics

This step takes the intermediate CPA-to-county match table from the spatial join and prepares it for the final county assignment.

After the previous cell, `cpa_county_matches` already tells us **which counties each CPA intersects**. However, that is not enough to decide the final county yet, because a single CPA can intersect multiple counties. To choose one **primary county**, the notebook needs to measure **how much** of the CPA lies inside each matched county.

## Add county geometry back into the match table

After a GeoPandas spatial join, the result keeps the geometry from the **left** dataset by default. In this case, that means `cpa_county_matches` still contains the CPA polygon geometry, but not the county polygon geometry.

To calculate overlap, the notebook brings the county geometry back in using `index_right`, which is the row index of the matched county from the county GeoDataFrame.

After this merge:
- `geometry` refers to the CPA polygon
- `county_geometry` refers to the matched county polygon

This gives the notebook both shapes needed to compute overlap.

## Compute the overlap metrics

Once both geometries are available, the notebook calculates three main quantities for each CPA-to-county match:

- **full CPA area**
  - the total area of the CPA polygon

- **overlap area**
  - the area where the CPA polygon and county polygon overlap

- **overlap share of the CPA**
  - the fraction of the CPA that lies inside that county

The notebook stores these as:
- `cpa_area_km2`
- `overlap_area_km2`
- `overlap_share_of_cpa`

These are the core measurements used in the next step to assign a **primary county** to each CPA.

## Why the area calculation is important

These area calculations work correctly because both the CPA polygons and county polygons were already reprojected to `EPSG:5070` in the previous step.

That matters because:
- latitude/longitude coordinates are fine for map display
- but they are not ideal for measuring area
- `EPSG:5070` is a U.S. equal-area projection, so the overlap calculations are much more reliable

## What the displayed table shows

The preview table shows the overlap results for the first few CPA-to-county matches.

The columns mean:

- `CPA_ID`
  - the Candidate Project Area identifier

- `county_fips`
  - the county FIPS code

- `county_name`
  - the short county name

- `county_name_full`
  - the fuller county name with legal/statistical description

- `state_fips`
  - the state FIPS code

- `overlap_area_km2`
  - how many square kilometers of the CPA fall inside that county

- `overlap_share_of_cpa`
  - what fraction of the CPA lies in that county

## How to interpret the example rows

Most of the example rows have `overlap_share_of_cpa = 1.0`, which means the entire CPA lies inside a single county. Those are very straightforward cases.

One row in the example has:
- `CPA_ID = 23`
- one match with **Skagit County** and `overlap_share_of_cpa = 0.172715`
- another match with **Whatcom County** and `overlap_share_of_cpa = 0.827285`

This means CPA 23 crosses a county boundary:
- about **17.3%** of the CPA lies in Skagit County
- about **82.7%** lies in Whatcom County

That is exactly the kind of case this step is designed to handle. In the next step, the notebook will assign **Whatcom County** as the primary county for CPA 23, because it has the **larger overlap area**.

In [ ]:
county_geom_lookup = counties[["geometry"]].rename(columns={"geometry": "county_geometry"}).copy()

cpa_county_matches = cpa_county_matches.merge(
    county_geom_lookup,
    left_on="index_right",
    right_index=True,
    how="left"
)

cpa_county_matches["cpa_area_m2"] = cpa_county_matches.geometry.area

cpa_county_matches["overlap_area_m2"] = cpa_county_matches.geometry.intersection(
    gpd.GeoSeries(cpa_county_matches["county_geometry"], crs=cpa.crs)
).area

cpa_county_matches["overlap_area_km2"] = cpa_county_matches["overlap_area_m2"] / 1_000_000

cpa_county_matches["cpa_area_km2"] = cpa_county_matches["cpa_area_m2"] / 1_000_000

cpa_county_matches["overlap_share_of_cpa"] = (
    cpa_county_matches["overlap_area_m2"] / cpa_county_matches["cpa_area_m2"]
)

display(
    cpa_county_matches[
        ["CPA_ID", "county_fips", "county_name", "county_name_full", "state_fips", "overlap_area_km2", "overlap_share_of_cpa"]
    ].head(20)
)

,CPA_ID,county_fips,county_name,county_name_full,state_fips,overlap_area_km2,overlap_share_of_cpa
0,1,53073,Whatcom,Whatcom County,53,2.000000,1.000000
1,2,53073,Whatcom,Whatcom County,53,3.750000,1.000000
2,3,53073,Whatcom,Whatcom County,53,2.934317,0.978106
3,4,53073,Whatcom,Whatcom County,53,2.000000,1.000000
4,6,53073,Whatcom,Whatcom County,53,2.250000,1.000000
5,7,53009,Clallam,Clallam County,53,2.750000,1.000000
6,8,53009,Clallam,Clallam County,53,3.250000,1.000000
7,10,53009,Clallam,Clallam County,53,2.250000,1.000000
8,12,53073,Whatcom,Whatcom County,53,2.250000,1.000000
9,13,53073,Whatcom,Whatcom County,53,2.000000,1.000000


## Assign one primary county to each CPA

At this point in the notebook, `cpa_county_matches` can contain more than one row for the same `CPA_ID`. That happens when a single Candidate Project Area crosses multiple county boundaries.

The purpose of this step is to reduce those many possible county matches down to **one final county assignment per CPA**. This is necessary because the downstream metadata file is expected to have a single county field for each CPA record.

## County assignment rule

The rule used here is:

- assign each CPA to the county with the **largest overlap area**

This means the notebook chooses the county that contains the largest share of the CPA polygon.

## How the code does this

The notebook first sorts the CPA-to-county match table by:
1. `CPA_ID`, so all rows for the same CPA are grouped together
2. `overlap_area_m2` in descending order, so the county with the largest overlap comes first for each CPA

After the table is sorted this way, the notebook keeps only the **first row for each `CPA_ID`**.

Because of the sorting, that first row is the county with the largest overlap area for that CPA. This is how the notebook turns the full set of possible county matches into one **primary county** per CPA.

## What fields are kept in the final county-assignment table

After selecting the best county for each CPA, the notebook keeps only the fields needed for the next steps:

- `CPA_ID`
- `county_fips`
- `county_name`
- `county_name_full`
- `state_fips`
- `cpa_area_km2`
- `overlap_area_km2`
- `overlap_share_of_cpa`

These fields preserve:
- the CPA identifier needed to merge back into `solar_lcoe_ReEDS.csv`
- the final county identifiers
- the overlap metrics that show how strong the county assignment was

## What the printed output means

The printed shape tells us the size of the new `primary_county` table.

At this point, the row count should be close to the number of unique `CPA_ID`s, because the notebook is now aiming for **one row per CPA** rather than one row per CPA-to-county match.

## What the displayed table shows

The displayed table is the first preview of the final one-county-per-CPA assignment.

Each row now represents:
- one CPA
- one assigned county
- the amount of the CPA that lies inside that county

The most important fields to interpret are:

- `CPA_ID`
  - the unique Candidate Project Area identifier

- `county_fips`
  - the county FIPS code for the selected county

- `county_name`
  - the short name of the selected county

- `county_name_full`
  - the full county name

- `state_fips`
  - the state FIPS code

- `cpa_area_km2`
  - the full size of the CPA in square kilometers

- `overlap_area_km2`
  - how much of that CPA lies inside the selected county

- `overlap_share_of_cpa`
  - the fraction of the CPA covered by the selected county

## How to interpret the overlap columns

If `overlap_share_of_cpa = 1.0`, that means the CPA lies entirely inside the selected county.

If `overlap_share_of_cpa` is less than 1.0, that means the CPA crosses county boundaries, but the selected county still contains the largest share of it.

This makes the overlap columns useful for QA, because they show whether a county assignment was very clear or more ambiguous.

## Why this step matters

This is the step where the notebook moves from:
- **all possible counties a CPA could belong to**

to:
- **one final county assignment per CPA**

That final one-county-per-CPA table is the main intermediate result that will be merged back into `solar_lcoe_ReEDS.csv` in the next step.

In [ ]:
primary_county = (
    cpa_county_matches.sort_values(
        ["CPA_ID", "overlap_area_m2"],
        ascending=[True, False]
    )
    .drop_duplicates(subset=["CPA_ID"], keep="first")
    .copy()
)

primary_county = primary_county[
    [
        "CPA_ID",
        "county_fips",
        "county_name",
        "county_name_full",
        "state_fips",
        "cpa_area_km2",
        "overlap_area_km2",
        "overlap_share_of_cpa",
    ]
].reset_index(drop=True)

print("primary_county shape:", primary_county.shape)

display(primary_county.head(20))

primary_county shape: (406110, 8)


,CPA_ID,county_fips,county_name,county_name_full,state_fips,cpa_area_km2,overlap_area_km2,overlap_share_of_cpa
0,1,53073,Whatcom,Whatcom County,53,2.00,2.000000,1.000000
1,2,53073,Whatcom,Whatcom County,53,3.75,3.750000,1.000000
2,3,53073,Whatcom,Whatcom County,53,3.00,2.934317,0.978106
3,4,53073,Whatcom,Whatcom County,53,2.00,2.000000,1.000000
4,6,53073,Whatcom,Whatcom County,53,2.25,2.250000,1.000000
5,7,53009,Clallam,Clallam County,53,2.75,2.750000,1.000000
6,8,53009,Clallam,Clallam County,53,3.25,3.250000,1.000000
7,10,53009,Clallam,Clallam County,53,2.25,2.250000,1.000000
8,12,53073,Whatcom,Whatcom County,53,2.25,2.250000,1.000000
9,13,53073,Whatcom,Whatcom County,53,2.00,2.000000,1.000000


## Add centroid longitude and latitude for each CPA

This step adds a simple point location for each Candidate Project Area after the county assignment has already been made.

The main county-assignment workflow is already complete at this point. Each CPA has already been matched to a primary county using polygon overlap. This cell does **not** change that county assignment. Instead, it adds centroid longitude and latitude as extra geographic reference fields.

## Why this step is included

The main deliverable is the county-enriched metadata file, so longitude and latitude are not the core requirement. However, they are still useful because:

- they provide a simple map-point representation for each CPA
- they make it easier to inspect or visualize the data later
- they may be useful as input features in downstream clustering or QA workflows

The important distinction is that these coordinates are added **after** the county assignment. They are not used to decide which county a CPA belongs to.

## What the code does

The code starts from the CPA polygon layer, where each row represents one solar Candidate Project Area and includes:
- `CPA_ID`
- `geometry`

It then computes the **centroid** of each CPA polygon.

A centroid is the geometric center point of a polygon. This gives each CPA a single representative point location.

The notebook then builds a new GeoDataFrame using:
- `CPA_ID` as the identifier
- the centroid point as the geometry

Because the CPA polygons were previously reprojected into `EPSG:5070` for area calculations, the centroid points are initially in that same projected coordinate system. The notebook then reprojects the centroids to `EPSG:4326`, which is standard latitude/longitude.

After that, it extracts:
- `longitude` from the x-coordinate
- `latitude` from the y-coordinate

Finally, it keeps only:
- `CPA_ID`
- `longitude`
- `latitude`

and merges those fields back into the one-county-per-CPA table.

## What the displayed table shows

The table is the updated `primary_county` table after the centroid coordinates have been added.

Each row still represents:
- one CPA
- one assigned primary county

But now the table also includes:
- `longitude`
- `latitude`

along with the county-assignment fields and overlap metrics.

## What the columns mean

The key fields shown in the table are:

- `CPA_ID`
  - the Candidate Project Area identifier

- `county_fips`
  - the county FIPS code of the assigned county

- `county_name`
  - the short county name

- `county_name_full`
  - the full county name

- `state_fips`
  - the state FIPS code

- `cpa_area_km2`
  - the total area of the CPA polygon

- `overlap_area_km2`
  - the area of the CPA that lies in the assigned county

- `overlap_share_of_cpa`
  - the fraction of the CPA that lies in the assigned county

- `longitude`
  - the centroid longitude of the CPA polygon

- `latitude`
  - the centroid latitude of the CPA polygon

## How to interpret the example rows

The first few rows show CPAs assigned to counties such as Whatcom County, Clallam County, and Skagit County in Washington.

For example:
- a row with `overlap_share_of_cpa = 1.0` means the CPA lies entirely in that county
- a row with `overlap_share_of_cpa = 0.827285` means about 82.7% of that CPA lies in the assigned county, while the rest lies in another county

The added longitude and latitude values give a single point location for each of those CPAs, which makes the table easier to map or inspect later.

## Why this step matters

This step makes the final CPA-to-county table more useful without changing the main county-assignment result.

It adds a simple geographic point representation for each CPA while preserving the more accurate polygon-based county assignment from the earlier steps.

In [ ]:
cpa_centroids = cpa.copy()

cpa_centroids["centroid_geometry"] = cpa_centroids.geometry.centroid

cpa_centroids = gpd.GeoDataFrame(
    cpa_centroids[["CPA_ID"]],
    geometry=cpa_centroids["centroid_geometry"],
    crs=cpa.crs
).to_crs("EPSG:4326")

cpa_centroids["longitude"] = cpa_centroids.geometry.x

cpa_centroids["latitude"] = cpa_centroids.geometry.y

cpa_centroids = cpa_centroids[["CPA_ID", "longitude", "latitude"]].copy()

primary_county = primary_county.merge(
    cpa_centroids,
    on="CPA_ID",
    how="left"
)

display(primary_county.head(20))

,CPA_ID,county_fips,county_name,county_name_full,state_fips,cpa_area_km2,overlap_area_km2,overlap_share_of_cpa,longitude,latitude
0,1,53073,Whatcom,Whatcom County,53,2.00,2.000000,1.000000,-122.304404,48.970641
1,2,53073,Whatcom,Whatcom County,53,3.75,3.750000,1.000000,-122.366630,48.938467
2,3,53073,Whatcom,Whatcom County,53,3.00,2.934317,0.978106,-121.965474,48.989242
3,4,53073,Whatcom,Whatcom County,53,2.00,2.000000,1.000000,-122.131763,48.911985
4,6,53073,Whatcom,Whatcom County,53,2.25,2.250000,1.000000,-122.137697,48.875921
5,7,53009,Clallam,Clallam County,53,2.75,2.750000,1.000000,-124.683608,48.350214
6,8,53009,Clallam,Clallam County,53,3.25,3.250000,1.000000,-124.610383,48.354214
7,10,53009,Clallam,Clallam County,53,2.25,2.250000,1.000000,-124.557623,48.299891
8,12,53073,Whatcom,Whatcom County,53,2.25,2.250000,1.000000,-122.466001,48.695710
9,13,53073,Whatcom,Whatcom County,53,2.00,2.000000,1.000000,-122.257214,48.712506


## Add clearer state fields and a plain county column

At this point in the notebook, `primary_county` already contains the core one-county-per-CPA assignment. Each row represents one Candidate Project Area and includes:
- `CPA_ID`
- county identifiers
- overlap metrics
- centroid longitude and latitude

The purpose of this step is to make that table easier to read and easier to use in downstream clustering workflows.

## Why this step is needed

The county shapefile already gave us `state_fips`, which is useful as a numeric identifier, but it is not very readable. For later QA, reporting, and clustering configuration, it is more helpful to also have:
- the full state name
- the two-letter state abbreviation
- a plain county field with a simple name

This step adds those cleaner fields without changing the county assignment itself.

## Where the new state fields come from

The notebook uses a file already stored in the repo:

- `county_layer_for_gui.geojson`

This file was used in earlier work and contains convenient county lookup fields such as:
- `GEOID`
- `STATE_NAME`
- `STUSPS`

The notebook uses this file as a lookup table keyed on county FIPS.

## How the lookup works

The code keeps only the fields needed from the GeoJSON:
- `GEOID`
- `STATE_NAME`
- `STUSPS`

Then it renames them into clearer names:
- `county_fips`
- `state_name`
- `state_abbrev`

The lookup table is then merged into `primary_county` using `county_fips`.

This means the state fields are being added based on the county that was already assigned in the previous step.

## What the merge adds

After this merge, each CPA row should now include:
- `county_fips`
- `county_name`
- `state_name`
- `state_abbrev`

This makes the table much easier to read because:
- `53` becomes `Washington`
- `WA` is available as the short abbreviation
- the table now has both machine-readable IDs and human-readable location fields

## Why the notebook adds a plain `county` column

The table already has `county_name`, but the notebook also creates a plain column called `county`.

This is done because the PowerGenome renewable clustering documentation refers to grouping on a field named `county`. Adding that column now makes the final output more directly usable later, without requiring another rename step.

So at the end of this cell:
- `county_name` is kept as the descriptive county field
- `county` is added as a simple clustering-ready county field

## What the displayed table shows

The output table is the updated `primary_county` table after the new state fields and the plain `county` column have been added.

The first screenshot shows the left side of the table, which already includes:
- `CPA_ID`
- county fields
- overlap metrics
- centroid longitude and latitude

The second screenshot shows the newly added fields on the right side:
- `state_name`
- `state_abbrev`
- `county`

## How to interpret the example rows

The example rows show CPAs assigned to counties in Washington.

For example:
- `state_name = Washington`
- `state_abbrev = WA`
- `county = Whatcom` or `Clallam` or `Skagit`

This means the notebook successfully translated the numeric county/state identifiers into cleaner text fields that are easier to understand and easier to use later.

## Why this step matters

This step does not change which county each CPA belongs to. That assignment was already decided in the previous step.

Instead, this step improves the table by making it:
- more readable for QA and presentation
- more usable in downstream PowerGenome / Switch workflows
- more directly compatible with clustering configs that expect a `county` field

This is the main intermediate table that will be merged back into `solar_lcoe_ReEDS.csv` in the next step.

In [ ]:
if COUNTY_GEOJSON_OPTIONAL.exists():
    county_geo = gpd.read_file(COUNTY_GEOJSON_OPTIONAL)

    county_geo = county_geo[["GEOID", "STATE_NAME", "STUSPS"]].drop_duplicates().rename(
        columns={
            "GEOID": "county_fips",
            "STATE_NAME": "state_name",
            "STUSPS": "state_abbrev",
        }
    )

    primary_county = primary_county.merge(
        county_geo,
        on="county_fips",
        how="left"
    )

primary_county["county"] = primary_county["county_name"]

display(primary_county.head(20))

,CPA_ID,county_fips,county_name,county_name_full,state_fips,cpa_area_km2,overlap_area_km2,overlap_share_of_cpa,longitude,latitude,state_name,state_abbrev,county
0,1,53073,Whatcom,Whatcom County,53,2.00,2.000000,1.000000,-122.304404,48.970641,Washington,WA,Whatcom
1,2,53073,Whatcom,Whatcom County,53,3.75,3.750000,1.000000,-122.366630,48.938467,Washington,WA,Whatcom
2,3,53073,Whatcom,Whatcom County,53,3.00,2.934317,0.978106,-121.965474,48.989242,Washington,WA,Whatcom
3,4,53073,Whatcom,Whatcom County,53,2.00,2.000000,1.000000,-122.131763,48.911985,Washington,WA,Whatcom
4,6,53073,Whatcom,Whatcom County,53,2.25,2.250000,1.000000,-122.137697,48.875921,Washington,WA,Whatcom
5,7,53009,Clallam,Clallam County,53,2.75,2.750000,1.000000,-124.683608,48.350214,Washington,WA,Clallam
6,8,53009,Clallam,Clallam County,53,3.25,3.250000,1.000000,-124.610383,48.354214,Washington,WA,Clallam
7,10,53009,Clallam,Clallam County,53,2.25,2.250000,1.000000,-124.557623,48.299891,Washington,WA,Clallam
8,12,53073,Whatcom,Whatcom County,53,2.25,2.250000,1.000000,-122.466001,48.695710,Washington,WA,Whatcom
9,13,53073,Whatcom,Whatcom County,53,2.00,2.000000,1.000000,-122.257214,48.712506,Washington,WA,Whatcom


## Merge the county assignment back into the solar metadata file

This step combines the two main pieces built earlier in the notebook:

- the original solar metadata table from `solar_lcoe_ReEDS.csv`
- the one-county-per-CPA lookup table in `primary_county`

The purpose of this step is to create a single final metadata table that keeps all of the original solar candidate-site attributes while also adding the new geographic fields needed for county-based clustering.

## What `solar_meta` is

`solar_meta` is the original metadata table from `solar_lcoe_ReEDS.csv`.

It contains one row per solar Candidate Project Area and includes the original project and cost-related variables used in the Switch-PG-ReEDS / PowerGenome workflow, such as:
- site size and shape fields
- land-use and environmental variables
- qualification flags
- technology type
- model region fields
- cost and interconnection variables

At this point in the notebook, `solar_meta` still does **not** yet contain the county assignment that was built through the geographic matching steps.

## What `primary_county` is

`primary_county` is the county-assignment table created in the earlier steps.

Each row in `primary_county` already contains:
- one `CPA_ID`
- one assigned county
- one assigned state
- overlap metrics showing how strong that county assignment was
- centroid longitude and latitude

So `primary_county` is the geographic lookup table, while `solar_meta` is the original solar metadata table.

## What the merge is doing

The merge joins `primary_county` back into `solar_meta` using `CPA_ID`.

It uses a **left merge**, which means:
- every original row in `solar_meta` is kept
- the county and state fields from `primary_county` are attached wherever the `CPA_ID` matches

This is important because the goal is not to replace the original solar metadata. The goal is to **preserve all of it** and enrich it with county information.

The result is a new table called `solar_meta_with_county`.

## What fields are added

After the merge, the final metadata table includes all original columns from `solar_lcoe_ReEDS.csv`, plus the new location fields created in the notebook, including:

- `county`
- `county_name`
- `county_name_full`
- `county_fips`
- `state_fips`
- `state_name`
- `state_abbrev`
- `cpa_area_km2`
- `overlap_area_km2`
- `overlap_share_of_cpa`
- `longitude`
- `latitude`

This is what turns the original solar metadata file into a county-enriched metadata file that can be used in the next clustering step.

## Why the plain `county` field is added

The notebook explicitly makes sure a field named `county` exists, even though the table already contains `county_name`.

This is done because the downstream PowerGenome renewable clustering documentation refers to grouping on a field named `county`. Adding that field here makes the final output more directly usable later, without requiring another rename step.

So the notebook now keeps both:
- `county_name` as a descriptive county label
- `county` as a clustering-ready field

## What the printed output means

The printed output gives a quick QA check on the merged result.

For example:

- `solar_meta_with_county shape: (405737, 55)`

This means:
- the merged metadata table has **405,737 rows**
- it has **55 columns** after the county/state fields and QA fields were added

The missing-share lines show the fraction of rows still missing county information:

- `Missing county_fips share`
- `Missing county_name share`
- `Missing county share`

These numbers are extremely small, which means the merge worked very well and nearly all rows received county information successfully.

## What the displayed tables show

The displayed output is the first preview of the fully enriched metadata table.

Because the table has many columns, the notebook output is visually spread across multiple sections.

From the screenshots, we can see several types of original metadata columns that were preserved, such as:

- physical and shape variables  
  - `Area`
  - `Shape_Leng`
  - `Shape_Area`

- distance and infrastructure variables  
  - `d_trans`
  - `d_sub`
  - `d_road`
  - `d_existing`
  - `d_plannedF`

- land and environmental variables  
  - `m_slope`
  - `m_popden`
  - `m_landcover`
  - `m_primeFarmland`

- qualification and screening fields  
  - `anyQual`
  - `Qual_Emp`
  - `Qual_Brown`
  - `Qual_noBF`

- modeling and cost fields  
  - `tech`
  - `metro_id`
  - `metro_region`
  - `cpa_mw`
  - `cf`
  - `resource_annuity`
  - `interconnect_annuity`
  - `lcoe`
  - `interconnect_capex_mw`
  - `total_interconnect_km`
  - `ipm_region`

The right side of the displayed output also shows the newly added county-enrichment fields, including:

- `county_fips`
- `county_name`
- `county_name_full`
- `state_fips`
- `cpa_area_km2`
- `overlap_area_km2`
- `overlap_share_of_cpa`
- `longitude`
- `latitude`
- `state_name`
- `state_abbrev`
- `county`

## How to interpret the example rows

The example rows show that the original solar metadata is still intact, but each CPA record now also has a geographic assignment attached to it.

For example, a row may now show:
- original cost and resource variables from `solar_lcoe_ReEDS.csv`
- `ipm_region = p1`
- `county_name = Whatcom`
- `state_name = Washington`
- `state_abbrev = WA`
- `longitude` and `latitude`
- `overlap_share_of_cpa = 1.0`

That means this CPA kept all of its original modeling information and now also includes a county/state assignment that can be used in downstream geographic grouping.

## Why this step matters

This is the key output-building step in the notebook.

Everything before this point was used to figure out **which county each CPA belongs to**. This step is where that result is actually attached back to the original ReEDS solar metadata.

After this merge, the notebook has produced the main county-enriched metadata table that will later be filtered to WECC states and saved as the final deliverable.

In [ ]:
solar_meta_with_county = solar_meta.merge(
    primary_county,
    on="CPA_ID",
    how="left"
)

if "county" not in solar_meta_with_county.columns:
    solar_meta_with_county["county"] = solar_meta_with_county["county_name"]

print("solar_meta_with_county shape:", solar_meta_with_county.shape)
print("Missing county_fips share:", solar_meta_with_county["county_fips"].isna().mean())
print("Missing county_name share:", solar_meta_with_county["county_name"].isna().mean())
print("Missing county share:", solar_meta_with_county["county"].isna().mean())

display(solar_meta_with_county.head())

solar_meta_with_county shape: (405737, 55)
Missing county_fips share: 2.464650746665944e-06
Missing county_name share: 2.464650746665944e-06
Missing county share: 2.464650746665944e-06


,Area,d_trans,d_sub,d_road,d_load_750,d_existing,d_plannedF,m_slope,m_popden,m_HMI,m_primeFarmland,incap,CPA_ID,m_aspect,m_aspect_min,Shape_Leng,m_landcover,exFacil,plFacil,Qual_Coal,Qual_Emp,Qual_Brown,anyQual,Qual_noBF,SocialImpa,EnviroImpa,Shape_Le_1,Shape_Area,pop_density_bin,tech,metro_id,metro_region,cpa_mw,cf,path,resource_annuity,resource_fom,interconnect_annuity,lcoe,interconnect_capex_mw,total_interconnect_km,offshore_interconnect_km,ipm_region,county_fips,county_name,county_name_full,state_fips,cpa_area_km2,overlap_area_km2,overlap_share_of_cpa,longitude,latitude,state_name,state_abbrev,county
0,2.00,19.142255,26.470939,0.000000,158.648693,265.038776,246.071077,0.125000,17.375000,0.780994,0.0,18.000001,1,48.625000,-1.0,6000.000177,81,0,0,0,1,0,1,1,3.906250,1.906250,6000.0,2000000.0,pop_den_(10-30],photovoltaic,42660,p1,4.500000,0.220500,"[1, 200355]",48359.91,15221.571,23691.982,45.182354,497798.70,143.821594,0.0,p1,53073,Whatcom,Whatcom County,53,2.00,2.000000,1.000000,-122.304404,48.970641,Washington,WA,Whatcom
1,3.75,13.030586,20.955390,1.085683,156.003598,264.714080,245.267324,1.600000,34.266667,0.786600,1.0,33.749998,2,145.866667,-1.0,8999.999866,81,0,0,0,1,0,1,1,-2.000000,5.250000,9000.0,3750000.0,pop_den_(30-40],photovoltaic,42660,p1,4.218750,0.224434,"[2, 200355]",48359.91,15221.571,23711.625,44.400440,498211.40,140.985809,0.0,p1,53073,Whatcom,Whatcom County,53,3.75,3.750000,1.000000,-122.366630,48.938467,Washington,WA,Whatcom
2,3.00,35.946548,45.514552,6.423029,156.322476,252.099160,235.024352,9.833333,0.000000,0.097131,0.0,26.999998,3,237.166667,129.0,10999.999725,42,0,0,0,1,0,1,1,-5.000000,1.000000,11000.0,3000000.0,pop_den_(0-5],photovoltaic,42660,p1,26.999998,0.222281,"[3, 200355]",48359.91,15221.571,25068.760,45.527447,526726.56,150.642654,0.0,p1,53073,Whatcom,Whatcom County,53,3.00,2.934317,0.978106,-121.965474,48.989242,Washington,WA,Whatcom
3,2.00,22.097498,30.668918,0.000000,149.529784,252.050029,233.665701,3.375000,35.125000,0.544276,1.0,18.000000,4,215.750000,-1.0,9000.000066,81,0,0,0,1,0,1,1,10.468750,9.593750,9000.0,2000000.0,pop_den_(30-40],photovoltaic,42660,p1,2.250000,0.223767,"[4, 200355]",48359.91,15221.571,22436.877,43.882450,471427.34,135.036041,0.0,p1,53073,Whatcom,Whatcom County,53,2.00,2.000000,1.000000,-122.131763,48.911985,Washington,WA,Whatcom
4,2.25,18.546952,28.017060,0.126662,145.881474,249.425370,230.785601,8.777778,1.888889,0.429502,0.0,20.250001,6,201.777778,7.0,8000.000037,42,0,0,0,1,0,1,1,9.583333,18.055556,8000.0,2250000.0,pop_den_(0-5],photovoltaic,42660,p1,20.250000,0.223767,"[6, 200355]",48359.91,15221.571,21848.154,43.582115,459057.50,130.457626,0.0,p1,53073,Whatcom,Whatcom County,53,2.25,2.250000,1.000000,-122.137697,48.875921,Washington,WA,Whatcom


## Filter the county-enriched outputs to the WECC states

At this point in the notebook, the county-enriched outputs still cover the full national CPA dataset. That is larger than needed for this project.

The purpose of this step is to keep only the records associated with the requested **WECC-state subset** so the final deliverables are focused on the Western Interconnection workflow and are easier to work with.

## What this filter is doing

The notebook applies the same state filter to three related tables:

- `primary_county`
  - the one-county-per-CPA assignment table

- `solar_meta_with_county`
  - the full county-enriched solar metadata table

- `cpa_county_matches`
  - the full CPA-to-county overlap table used for QA and auditing

This creates smaller WECC-only versions of all three tables:
- `primary_county_wecc`
- `solar_meta_with_county_wecc`
- `cpa_county_matches_wecc`

## Important detail about the filter

This is a **state-based filter**, not an exact transmission-footprint filter.

That means the notebook keeps all records in the listed states, even though some states such as Texas and South Dakota are only partially within the Western Interconnection. This matches the requested deliverable, which was to keep the specified WECC-state list.

## What the filtered states list means

The printed line:

- `WECC states present in filtered output: [...]`

is a QA check showing exactly which states remain after filtering.

From the output shown, the retained states are:
- Arizona
- California
- Colorado
- Idaho
- Montana
- Nevada
- New Mexico
- Oregon
- South Dakota
- Texas
- Utah
- Washington
- Wyoming

This confirms that the filter was applied correctly to the final output subset.

## What the printed shape comparisons mean

The shape outputs compare the full national tables to the WECC-only filtered tables.

For example:

- `primary_county full shape: (406110, 13)`
- `primary_county WECC-only shape: (184863, 13)`

This means:
- the full one-county-per-CPA table had **406,110 rows**
- the WECC-only version had **184,863 rows**

The same logic applies to the other two comparisons:

- `solar_meta_with_county full shape: (405737, 55)`
- `solar_meta_with_county WECC-only shape: (184776, 55)`

This means:
- the full county-enriched metadata table had **405,737 rows**
- the WECC-only version had **184,776 rows**

And:

- `cpa_county_matches full shape: (470356, 13)`
- `cpa_county_matches WECC-only shape: (205824, 13)`

This means:
- the full CPA-to-county overlap table had **470,356 rows**
- the WECC-only QA version had **205,824 rows**

Overall, these comparisons show that the filter substantially reduced the size of the outputs while preserving the same structure and columns.

## Why the row counts are different across the three tables

The three filtered tables do not have the same number of rows because they represent different levels of the workflow:

- `primary_county_wecc`
  - one assigned county per CPA

- `solar_meta_with_county_wecc`
  - one metadata row per CPA in the solar metadata file

- `cpa_county_matches_wecc`
  - all CPA-to-county overlap matches before choosing the primary county

That is why `cpa_county_matches_wecc` is still larger than the other two: it includes multiple county matches for some CPAs.

## What the displayed tables show

The displayed previews show the first few rows of the filtered WECC-only outputs.

The first displayed table shows the filtered `primary_county_wecc` table. This includes:
- `CPA_ID`
- county fields
- state fields
- overlap metrics
- centroid coordinates

This confirms that the county-assignment information was preserved after filtering.

The next displayed tables show the filtered `solar_meta_with_county_wecc` table. Because the table has many columns, the notebook output wraps it across several views. Together, these views show that the WECC-only metadata still includes:

- the original solar metadata variables from `solar_lcoe_ReEDS.csv`
- the added county and state fields
- the overlap QA fields
- the centroid longitude and latitude

The final screenshot section shows fields such as:
- `longitude`
- `latitude`
- `state_name`
- `state_abbrev`
- `county`

This is important because it confirms that the final filtered metadata is still clustering-ready after the state filter is applied.

## Why this step matters

This step does not change the county assignment logic or the metadata itself. Instead, it narrows the deliverables to the regional subset that is relevant for the project.

After this step, all of the main outputs are:
- county-enriched
- state-enriched
- filtered to the requested WECC-state subset
- smaller and more practical to save, review, and use downstream

This is the last major transformation before the final output files are written to disk.

In [ ]:
WECC_STATES = [
    "Arizona",
    "California",
    "Colorado",
    "Idaho",
    "Montana",
    "Nevada",
    "New Mexico",
    "Oregon",
    "South Dakota",
    "Texas",
    "Utah",
    "Washington",
    "Wyoming",
]

primary_county_wecc = primary_county[
    primary_county["state_name"].isin(WECC_STATES)
].copy()

solar_meta_with_county_wecc = solar_meta_with_county[
    solar_meta_with_county["state_name"].isin(WECC_STATES)
].copy()

cpa_county_matches_wecc = cpa_county_matches[
    cpa_county_matches["CPA_ID"].isin(primary_county_wecc["CPA_ID"])
].copy()

print("primary_county full shape:", primary_county.shape)
print("primary_county WECC-only shape:", primary_county_wecc.shape)

print("\nsolar_meta_with_county full shape:", solar_meta_with_county.shape)
print("solar_meta_with_county WECC-only shape:", solar_meta_with_county_wecc.shape)

print("\ncpa_county_matches full shape:", cpa_county_matches.shape)
print("cpa_county_matches WECC-only shape:", cpa_county_matches_wecc.shape)

print("\nWECC states present in filtered output:")
print(sorted(solar_meta_with_county_wecc["state_name"].dropna().unique().tolist()))

display(primary_county_wecc.head())
display(solar_meta_with_county_wecc.head())

primary_county full shape: (406110, 13)
primary_county WECC-only shape: (184863, 13)

solar_meta_with_county full shape: (405737, 55)
solar_meta_with_county WECC-only shape: (184776, 55)

cpa_county_matches full shape: (470356, 13)
cpa_county_matches WECC-only shape: (205824, 13)

WECC states present in filtered output:
['Arizona', 'California', 'Colorado', 'Idaho', 'Montana', 'Nevada', 'New Mexico', 'Oregon', 'South Dakota', 'Texas', 'Utah', 'Washington', 'Wyoming']


,CPA_ID,county_fips,county_name,county_name_full,state_fips,cpa_area_km2,overlap_area_km2,overlap_share_of_cpa,longitude,latitude,state_name,state_abbrev,county
0,1,53073,Whatcom,Whatcom County,53,2.00,2.000000,1.000000,-122.304404,48.970641,Washington,WA,Whatcom
1,2,53073,Whatcom,Whatcom County,53,3.75,3.750000,1.000000,-122.366630,48.938467,Washington,WA,Whatcom
2,3,53073,Whatcom,Whatcom County,53,3.00,2.934317,0.978106,-121.965474,48.989242,Washington,WA,Whatcom
3,4,53073,Whatcom,Whatcom County,53,2.00,2.000000,1.000000,-122.131763,48.911985,Washington,WA,Whatcom
4,6,53073,Whatcom,Whatcom County,53,2.25,2.250000,1.000000,-122.137697,48.875921,Washington,WA,Whatcom


,Area,d_trans,d_sub,d_road,d_load_750,d_existing,d_plannedF,m_slope,m_popden,m_HMI,m_primeFarmland,incap,CPA_ID,m_aspect,m_aspect_min,Shape_Leng,m_landcover,exFacil,plFacil,Qual_Coal,Qual_Emp,Qual_Brown,anyQual,Qual_noBF,SocialImpa,EnviroImpa,Shape_Le_1,Shape_Area,pop_density_bin,tech,metro_id,metro_region,cpa_mw,cf,path,resource_annuity,resource_fom,interconnect_annuity,lcoe,interconnect_capex_mw,total_interconnect_km,offshore_interconnect_km,ipm_region,county_fips,county_name,county_name_full,state_fips,cpa_area_km2,overlap_area_km2,overlap_share_of_cpa,longitude,latitude,state_name,state_abbrev,county
0,2.00,19.142255,26.470939,0.000000,158.648693,265.038776,246.071077,0.125000,17.375000,0.780994,0.0,18.000001,1,48.625000,-1.0,6000.000177,81,0,0,0,1,0,1,1,3.906250,1.906250,6000.0,2000000.0,pop_den_(10-30],photovoltaic,42660,p1,4.500000,0.220500,"[1, 200355]",48359.91,15221.571,23691.982,45.182354,497798.70,143.821594,0.0,p1,53073,Whatcom,Whatcom County,53,2.00,2.000000,1.000000,-122.304404,48.970641,Washington,WA,Whatcom
1,3.75,13.030586,20.955390,1.085683,156.003598,264.714080,245.267324,1.600000,34.266667,0.786600,1.0,33.749998,2,145.866667,-1.0,8999.999866,81,0,0,0,1,0,1,1,-2.000000,5.250000,9000.0,3750000.0,pop_den_(30-40],photovoltaic,42660,p1,4.218750,0.224434,"[2, 200355]",48359.91,15221.571,23711.625,44.400440,498211.40,140.985809,0.0,p1,53073,Whatcom,Whatcom County,53,3.75,3.750000,1.000000,-122.366630,48.938467,Washington,WA,Whatcom
2,3.00,35.946548,45.514552,6.423029,156.322476,252.099160,235.024352,9.833333,0.000000,0.097131,0.0,26.999998,3,237.166667,129.0,10999.999725,42,0,0,0,1,0,1,1,-5.000000,1.000000,11000.0,3000000.0,pop_den_(0-5],photovoltaic,42660,p1,26.999998,0.222281,"[3, 200355]",48359.91,15221.571,25068.760,45.527447,526726.56,150.642654,0.0,p1,53073,Whatcom,Whatcom County,53,3.00,2.934317,0.978106,-121.965474,48.989242,Washington,WA,Whatcom
3,2.00,22.097498,30.668918,0.000000,149.529784,252.050029,233.665701,3.375000,35.125000,0.544276,1.0,18.000000,4,215.750000,-1.0,9000.000066,81,0,0,0,1,0,1,1,10.468750,9.593750,9000.0,2000000.0,pop_den_(30-40],photovoltaic,42660,p1,2.250000,0.223767,"[4, 200355]",48359.91,15221.571,22436.877,43.882450,471427.34,135.036041,0.0,p1,53073,Whatcom,Whatcom County,53,2.00,2.000000,1.000000,-122.131763,48.911985,Washington,WA,Whatcom
4,2.25,18.546952,28.017060,0.126662,145.881474,249.425370,230.785601,8.777778,1.888889,0.429502,0.0,20.250001,6,201.777778,7.0,8000.000037,42,0,0,0,1,0,1,1,9.583333,18.055556,8000.0,2250000.0,pop_den_(0-5],photovoltaic,42660,p1,20.250000,0.223767,"[6, 200355]",48359.91,15221.571,21848.154,43.582115,459057.50,130.457626,0.0,p1,53073,Whatcom,Whatcom County,53,2.25,2.250000,1.000000,-122.137697,48.875921,Washington,WA,Whatcom


## Save the final WECC-only deliverables and create a run summary

This is the final output step of the notebook.

By this point, the notebook has already:
- assigned one primary county to each CPA
- merged that county assignment back into the solar metadata file
- filtered the outputs down to the requested WECC-state subset

The purpose of this step is to save those final WECC-only tables to disk and create a small summary file that records the most important QA results from the run.

## Which tables are being saved

The notebook saves three main WECC-only tables:

- `primary_county_wecc`
  - the one-county-per-CPA assignment table for the WECC subset

- `solar_meta_with_county_wecc`
  - the main final metadata file
  - this is the original `solar_lcoe_ReEDS.csv` plus the added county/state fields, overlap metrics, and centroid coordinates, filtered to the WECC states

- `cpa_county_matches_wecc`
  - the full CPA-to-county overlap table for the WECC subset
  - this is mainly a QA and audit table because it keeps all intersecting county matches before the notebook reduces them to one primary county per CPA

## What each saved file is for

The saved files serve slightly different purposes:

- **`cpa_primary_county_assignment.csv`**
  - compact reference table
  - shows the final assigned county for each CPA

- **`solar_lcoe_ReEDS_with_county.csv`**
  - the main deliverable
  - this is the county-enriched solar metadata file intended for downstream use

- **`solar_lcoe_ReEDS_with_county_clustering_ready.csv`**
  - same content as the main metadata file
  - saved with a clearer name to show that it is ready for the next clustering step

- **`cpa_county_overlap_full.parquet`**
  - QA and audit file
  - keeps the full CPA-to-county overlap results before final county selection
  - saved as parquet because it is more efficient for larger tables

## Why the overlap file is saved as parquet

The overlap table can be large because a single CPA may intersect more than one county. Saving it as parquet keeps the file size more manageable and preserves data types better than CSV.

The notebook also drops the raw geometry columns from this file before saving it. That keeps the audit file smaller while preserving the most important tabular overlap information.

## What the run summary is doing

After saving the tables, the notebook builds a small dictionary called `summary`.

This summary records the most important information from the run, including:
- the full national row counts
- the WECC-only row counts
- whether county fields are missing in the final WECC output
- whether important fields like state name, state abbreviation, longitude, and latitude are present
- the county assignment rule used
- the list of filtered states kept in the final output

The summary is then saved as:
- `run_summary.json`

This file is useful because it gives a quick QA snapshot of the entire workflow without needing to inspect the full CSV outputs.

## What the displayed summary means

The dictionary shown in the screenshot is the final QA summary from the run.

The key fields mean:

- `solar_meta_rows_full: 405737`
  - the full national solar metadata file has 405,737 rows before WECC filtering

- `solar_meta_rows_wecc_only: 184776`
  - the final WECC-only metadata file has 184,776 rows

- `solar_meta_unique_cpa_ids_full: 405737`
  - there are 405,737 unique CPA IDs in the full metadata table

- `solar_meta_unique_cpa_ids_wecc_only: 184776`
  - there are 184,776 unique CPA IDs in the WECC-only metadata table

- `cpa_polygon_rows: 406110`
  - the solar CPA shapefile contained 406,110 CPA polygon rows

- `county_rows: 3235`
  - the county shapefile contained 3,235 county polygons

- `primary_county_rows_wecc_only: 184863`
  - the WECC-only primary county table contains 184,863 CPA assignments

- `missing_county_fips_share_wecc_only: 0.0`
- `missing_county_name_share_wecc_only: 0.0`
- `missing_county_share_wecc_only: 0.0`
  - these show that the final WECC-only metadata file has essentially no missing county information

- `has_state_name: True`
- `has_state_abbrev: True`
- `has_longitude: True`
- `has_latitude: True`
  - these confirm that the expected enrichment fields were successfully added

- `county_assignment_method: 'largest_overlap_area'`
  - this records the exact rule used to assign one primary county per CPA

- `filtered_states: [...]`
  - this lists the states retained in the WECC-only outputs

## Why this cell matters

This cell turns the notebook from an analysis workflow into a completed deliverable.

It does three important things:
1. saves the final WECC-only output files
2. saves the QA and audit files
3. records a compact summary showing that the workflow completed successfully

In other words, this is the final handoff step. After this cell runs, the notebook has produced the actual files that can be shared and used in downstream clustering work.

In [ ]:
primary_county_wecc.to_csv(
    OUTPUT_DIR / "cpa_primary_county_assignment.csv",
    index=False
)

solar_meta_with_county_wecc.to_csv(
    OUTPUT_DIR / "solar_lcoe_ReEDS_with_county.csv",
    index=False
)

solar_meta_with_county_wecc.to_csv(
    OUTPUT_DIR / "solar_lcoe_ReEDS_with_county_clustering_ready.csv",
    index=False
)

cpa_county_matches_wecc.drop(columns=["geometry", "county_geometry"]).to_parquet(
    OUTPUT_DIR / "cpa_county_overlap_full.parquet",
    index=False
)

summary = {
    "solar_meta_rows_full": int(len(solar_meta)),
    "solar_meta_rows_wecc_only": int(len(solar_meta_with_county_wecc)),
    "solar_meta_unique_cpa_ids_full": int(solar_meta["CPA_ID"].nunique()),
    "solar_meta_unique_cpa_ids_wecc_only": int(solar_meta_with_county_wecc["CPA_ID"].nunique()),
    "cpa_polygon_rows": int(len(cpa)),
    "county_rows": int(len(counties)),
    "primary_county_rows_wecc_only": int(len(primary_county_wecc)),
    "missing_county_fips_share_wecc_only": float(solar_meta_with_county_wecc["county_fips"].isna().mean()),
    "missing_county_name_share_wecc_only": float(solar_meta_with_county_wecc["county_name"].isna().mean()),
    "missing_county_share_wecc_only": float(solar_meta_with_county_wecc["county"].isna().mean()),
    "has_state_name": "state_name" in solar_meta_with_county_wecc.columns,
    "has_state_abbrev": "state_abbrev" in solar_meta_with_county_wecc.columns,
    "has_longitude": "longitude" in solar_meta_with_county_wecc.columns,
    "has_latitude": "latitude" in solar_meta_with_county_wecc.columns,
    "county_assignment_method": "largest_overlap_area",
    "filtered_states": WECC_STATES,
}

with open(OUTPUT_DIR / "run_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

summary

{'solar_meta_rows_full': 405737,
 'solar_meta_rows_wecc_only': 184776,
 'solar_meta_unique_cpa_ids_full': 405737,
 'solar_meta_unique_cpa_ids_wecc_only': 184776,
 'cpa_polygon_rows': 406110,
 'county_rows': 3235,
 'primary_county_rows_wecc_only': 184863,
 'missing_county_fips_share_wecc_only': 0.0,
 'missing_county_name_share_wecc_only': 0.0,
 'missing_county_share_wecc_only': 0.0,
 'has_state_name': True,
 'has_state_abbrev': True,
 'has_longitude': True,
 'has_latitude': True,
 'county_assignment_method': 'largest_overlap_area',
 'filtered_states': ['Arizona',
  'California',
  'Colorado',
  'Idaho',
  'Montana',
  'Nevada',
  'New Mexico',
  'Oregon',
  'South Dakota',
  'Texas',
  'Utah',
  'Washington',
  'Wyoming']}

## **What this notebook produces**

The main output from this notebook is:

- `data/reeds_county_mapping/outputs/solar_lcoe_ReEDS_with_county.csv`

This is the updated solar metadata file with county information added for each `CPA_ID`, filtered to the requested WECC states. It is the main file to use for the next clustering step.

A second copy is also saved as:

- `data/reeds_county_mapping/outputs/solar_lcoe_ReEDS_with_county_clustering_ready.csv`

This file contains the same data, but with a name that makes its intended use clearer. It includes all original columns from `solar_lcoe_ReEDS.csv`, along with the added county and state fields, overlap metrics, and centroid longitude/latitude.

The notebook also saves two QA files:

- `data/reeds_county_mapping/outputs/cpa_primary_county_assignment.csv`
- `data/reeds_county_mapping/outputs/cpa_county_overlap_full.parquet`

`cpa_primary_county_assignment.csv` contains one final county assignment per `CPA_ID` for the filtered WECC-state subset. `cpa_county_overlap_full.parquet` contains the full CPA-to-county overlap table for the same filtered subset before selecting the primary county.

The main deliverable is the WECC-filtered, county-enriched `solar_lcoe_ReEDS_with_county.csv`.

### **Ignore: Final Data Checks** 

In [34]:
# Confirm one row per CPA_ID in the final WECC-only metadata file
print("total rows:", len(solar_meta_with_county_wecc))
print("unique CPA_IDs:", solar_meta_with_county_wecc["CPA_ID"].nunique())
print("duplicate CPA_ID rows:", solar_meta_with_county_wecc.duplicated(subset=["CPA_ID"]).sum())

total rows: 184776
unique CPA_IDs: 184776
duplicate CPA_ID rows: 0


In [35]:
# Show any rows still missing county fields
missing_rows = solar_meta_with_county_wecc[
    solar_meta_with_county_wecc["county"].isna()
    | solar_meta_with_county_wecc["county_name"].isna()
    | solar_meta_with_county_wecc["county_fips"].isna()
]

print("rows with any missing county info:", len(missing_rows))
display(missing_rows.head(20))

rows with any missing county info: 0


,Area,d_trans,d_sub,d_road,d_load_750,d_existing,d_plannedF,m_slope,m_popden,m_HMI,m_primeFarmland,incap,CPA_ID,m_aspect,m_aspect_min,Shape_Leng,m_landcover,exFacil,plFacil,Qual_Coal,Qual_Emp,Qual_Brown,anyQual,Qual_noBF,SocialImpa,EnviroImpa,Shape_Le_1,Shape_Area,pop_density_bin,tech,metro_id,metro_region,cpa_mw,cf,path,resource_annuity,resource_fom,interconnect_annuity,lcoe,interconnect_capex_mw,total_interconnect_km,offshore_interconnect_km,ipm_region,county_fips,county_name,county_name_full,state_fips,cpa_area_km2,overlap_area_km2,overlap_share_of_cpa,longitude,latitude,state_name,state_abbrev,county


In [36]:
# Inspect potentially ambiguous county assignments
low_overlap = primary_county_wecc[
    primary_county_wecc["overlap_share_of_cpa"] < 0.5
].sort_values("overlap_share_of_cpa")

print("low-overlap CPA assignments:", len(low_overlap))
display(low_overlap.head(20))

low-overlap CPA assignments: 196


,CPA_ID,county_fips,county_name,county_name_full,state_fips,cpa_area_km2,overlap_area_km2,overlap_share_of_cpa,longitude,latitude,state_name,state_abbrev,county
96676,100642,48465,Val Verde,Val Verde County,48,4.000000,0.000376,0.000094,-101.197199,29.513047,Texas,TX,Val Verde
178350,184003,48323,Maverick,Maverick County,48,4.250000,0.001839,0.000433,-100.521647,28.740355,Texas,TX,Maverick
96737,100703,48479,Webb,Webb County,48,2.250000,0.001169,0.000520,-99.602522,27.633033,Texas,TX,Webb
182837,188490,48505,Zapata,Zapata County,48,5.500000,0.004050,0.000736,-99.339943,26.911892,Texas,TX,Zapata
182838,188491,48505,Zapata,Zapata County,48,4.500000,0.005705,0.001268,-99.218423,26.717322,Texas,TX,Zapata
96742,100708,48505,Zapata,Zapata County,48,3.500000,0.005961,0.001703,-99.248942,26.782915,Texas,TX,Zapata
183959,189612,48479,Webb,Webb County,48,9.199933,0.040332,0.004384,-99.525019,27.342099,Texas,TX,Webb
184041,189694,48479,Webb,Webb County,48,7.300401,0.065814,0.009015,-99.503867,27.409756,Texas,TX,Webb
96753,100719,48061,Cameron,Cameron County,48,3.500000,0.106197,0.030342,-97.555764,25.924918,Texas,TX,Cameron
180531,186184,48323,Maverick,Maverick County,48,5.500000,0.173456,0.031537,-100.363822,28.471060,Texas,TX,Maverick


In [37]:
# Confirm only the requested states remain
print(sorted(solar_meta_with_county_wecc["state_name"].dropna().unique().tolist()))

['Arizona', 'California', 'Colorado', 'Idaho', 'Montana', 'Nevada', 'New Mexico', 'Oregon', 'South Dakota', 'Texas', 'Utah', 'Washington', 'Wyoming']


In [38]:
# Summarize how much of the national file remains after WECC filtering
print("full rows:", len(solar_meta_with_county))
print("WECC-only rows:", len(solar_meta_with_county_wecc))
print("share kept:", len(solar_meta_with_county_wecc) / len(solar_meta_with_county))

full rows: 405737
WECC-only rows: 184776
share kept: 0.4554083063659464


In [ ]:
# QA check 1: compare CPA_ID coverage across the metadata file and shapefile

# This checks whether there are CPA_IDs that appear:
# - in the original solar metadata file but not in the CPA polygon shapefile
# - in the CPA polygon shapefile but not in the metadata file
#
# Small differences are not automatically an error, but this helps explain
# row-count mismatches and gives a clearer picture of coverage.

solar_meta_ids = set(solar_meta["CPA_ID"].dropna().unique())
cpa_ids = set(cpa["CPA_ID"].dropna().unique())

only_in_metadata = solar_meta_ids - cpa_ids
only_in_shapefile = cpa_ids - solar_meta_ids

print("unique CPA_IDs in solar_meta:", len(solar_meta_ids))
print("unique CPA_IDs in CPA shapefile:", len(cpa_ids))
print("CPA_IDs only in solar_meta:", len(only_in_metadata))
print("CPA_IDs only in shapefile:", len(only_in_shapefile))

print("\nExample CPA_IDs only in solar_meta:")
print(sorted(list(only_in_metadata))[:20])

print("\nExample CPA_IDs only in shapefile:")
print(sorted(list(only_in_shapefile))[:20])

unique CPA_IDs in solar_meta: 405737
unique CPA_IDs in CPA shapefile: 406110
CPA_IDs only in solar_meta: 0
CPA_IDs only in shapefile: 373

Example CPA_IDs only in solar_meta:
[]

Example CPA_IDs only in shapefile:
[np.int64(33), np.int64(1838), np.int64(7656), np.int64(7661), np.int64(14379), np.int64(15419), np.int64(15441), np.int64(15442), np.int64(15458), np.int64(15459), np.int64(15470), np.int64(15476), np.int64(15477), np.int64(22358), np.int64(22359), np.int64(22377), np.int64(22383), np.int64(22384), np.int64(22385), np.int64(22411)]


In [ ]:
# QA check 2: summarize how confident the county assignments look

# overlap_share_of_cpa tells us what fraction of the CPA polygon lies in the
# assigned primary county.
#
# Values near 1.0 mean the county assignment is very clear.
# Lower values mean the CPA crosses county boundaries more substantially.

overlap_summary = primary_county_wecc["overlap_share_of_cpa"].describe()
print("Overlap-share summary:")
display(overlap_summary)

# Bucket the overlap shares into easy-to-read ranges.
overlap_buckets = pd.cut(
    primary_county_wecc["overlap_share_of_cpa"],
    bins=[0, 0.25, 0.5, 0.75, 0.9, 0.99, 1.0],
    include_lowest=True
).value_counts().sort_index()

print("\nOverlap-share buckets:")
display(overlap_buckets)

Overlap-share summary:


count    184863.000000
mean          0.977511
std           0.083319
min           0.000094
25%           1.000000
50%           1.000000
75%           1.000000
max           1.000000
Name: overlap_share_of_cpa, dtype: float64


Overlap-share buckets:


overlap_share_of_cpa
(-0.001, 0.25]        21
(0.25, 0.5]          175
(0.5, 0.75]         7680
(0.75, 0.9]         5234
(0.9, 0.99]         5146
(0.99, 1.0]       141022
Name: count, dtype: int64

In [ ]:
# QA check 3: save potentially ambiguous county assignments

# These are the cases where the assigned county covers less than half of the CPA.
# They are not necessarily wrong, but they are the rows most worth reviewing
# if someone wants to inspect edge cases later.

ambiguous_cpas = primary_county_wecc[
    primary_county_wecc["overlap_share_of_cpa"] < 0.5
].sort_values("overlap_share_of_cpa")

print("Potentially ambiguous CPA assignments:", len(ambiguous_cpas))
display(ambiguous_cpas.head(20))

# Save them as a separate QA file for review.
ambiguous_cpas.to_csv(
    OUTPUT_DIR / "cpa_primary_county_assignment_ambiguous_overlap_lt_50pct.csv",
    index=False
)

Potentially ambiguous CPA assignments: 196


,CPA_ID,county_fips,county_name,county_name_full,state_fips,cpa_area_km2,overlap_area_km2,overlap_share_of_cpa,longitude,latitude,state_name,state_abbrev,county
96676,100642,48465,Val Verde,Val Verde County,48,4.000000,0.000376,0.000094,-101.197199,29.513047,Texas,TX,Val Verde
178350,184003,48323,Maverick,Maverick County,48,4.250000,0.001839,0.000433,-100.521647,28.740355,Texas,TX,Maverick
96737,100703,48479,Webb,Webb County,48,2.250000,0.001169,0.000520,-99.602522,27.633033,Texas,TX,Webb
182837,188490,48505,Zapata,Zapata County,48,5.500000,0.004050,0.000736,-99.339943,26.911892,Texas,TX,Zapata
182838,188491,48505,Zapata,Zapata County,48,4.500000,0.005705,0.001268,-99.218423,26.717322,Texas,TX,Zapata
96742,100708,48505,Zapata,Zapata County,48,3.500000,0.005961,0.001703,-99.248942,26.782915,Texas,TX,Zapata
183959,189612,48479,Webb,Webb County,48,9.199933,0.040332,0.004384,-99.525019,27.342099,Texas,TX,Webb
184041,189694,48479,Webb,Webb County,48,7.300401,0.065814,0.009015,-99.503867,27.409756,Texas,TX,Webb
96753,100719,48061,Cameron,Cameron County,48,3.500000,0.106197,0.030342,-97.555764,25.924918,Texas,TX,Cameron
180531,186184,48323,Maverick,Maverick County,48,5.500000,0.173456,0.031537,-100.363822,28.471060,Texas,TX,Maverick
